# 02 — Output Parsers & Structured Output
### Turning raw model text into validated Python objects

A chat model's raw output is always text. But `classify_ticket()` in
SupportPilot needs a `TicketClassification` object with an enum-constrained
`issue_type`, a `confidence` float between 0 and 1, etc. — the same
Week 2 discipline ("output validation with Pydantic") from the course applies
here, just wired through LangChain instead of raw function calling.

## 2.1 The problem: raw text isn't usable

If you just ask a model to classify a ticket in plain English, you get plain
English back — not something your escalation logic can branch on.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the support ticket."),
    ("human", "{ticket_text}"),
])

# A fake model returning a free-text answer, the way an unconstrained
# real model might.
unstructured_model = FakeListChatModel(responses=[
    "This seems to be about a refund. The customer wants their money back. "
    "I'd say this is fairly urgent."
])

chain = prompt | unstructured_model
result = chain.invoke({"ticket_text": "I want a refund for my order."})
print(result.content)
print()
print("Notice: no field you can reliably branch code on. issue_type=? confidence=?")


## 2.2 Fix 1 — `StrOutputParser` (simplest case)

Sometimes you *do* just want the text, cleanly, without the `AIMessage`
wrapper. `StrOutputParser` extracts `.content` for you — this is exactly
what `chains.py`'s drafting chain uses, because a drafted customer response
is meant to stay as free text, not be forced into a schema.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

draft_chain = prompt | unstructured_model | StrOutputParser()
draft_text = draft_chain.invoke({"ticket_text": "I want a refund for my order."})
print(type(draft_text))   # plain str now, not an AIMessage
print(draft_text)


## 2.3 Fix 2 — structured output with Pydantic

For the *classification* and *validation* steps, we need actual structure.
LangChain gives you two approaches:

1. **`with_structured_output(PydanticModel)`** — the model is asked (usually
   via native tool-calling) to return arguments matching your schema
   directly. This is what `chains.py` uses in `live` mode, because
   `ChatAnthropic` supports tool calling.
2. **`PydanticOutputParser`** — you inject formatting instructions into the
   prompt yourself, the model returns a JSON string, and the parser validates
   and converts it. Useful with models that don't support tool calling, or
   when you want to see exactly what instructions are being injected.

Let's do approach 2 first since it works with a fake model (no tool-calling
support needed) and makes the mechanism visible.

In [ ]:
from models import TicketClassification   # the real Pydantic model from the project
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=TicketClassification)

# This is what the parser injects into your prompt so the model knows the
# exact JSON shape to return. Worth reading once.
print(parser.get_format_instructions()[:600], "...")


In [ ]:
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the ticket. {format_instructions}"),
    ("human", "ticket_id: {ticket_id}\nticket_text: {ticket_text}"),
]).partial(format_instructions=parser.get_format_instructions())

# A fake model this time returns a JSON string matching TicketClassification.
# In real life the actual model would produce this JSON itself, guided by
# the format_instructions above.
fake_json_model = FakeListChatModel(responses=[
    '{"ticket_id": "T002", "issue_type": "refund_request", "urgency": "medium", '
    '"sentiment": "neutral", "confidence": 0.85, "extracted_order_id": null, '
    '"extracted_customer_id": null, "summary": "Customer wants a refund"}'
])

structured_chain = structured_prompt | fake_json_model | parser
result = structured_chain.invoke({"ticket_id": "T002", "ticket_text": "I want a refund."})

print(type(result))          # TicketClassification, a real Pydantic object
print(result.issue_type)     # IssueType.REFUND_REQUEST -- an actual enum value
print(result.confidence)
print(result.model_dump())   # convert back to a plain dict when you need one


Try passing malformed JSON or an invalid enum value through the same parser
and see what happens — Pydantic validation will raise, exactly like it would
in `pipeline.py`'s guarded `TicketClassification(**classification)` call
after the classification chain runs.

In [ ]:
bad_model = FakeListChatModel(responses=[
    '{"ticket_id": "T003", "issue_type": "not_a_real_category", "urgency": "medium", '
    '"sentiment": "neutral", "confidence": 0.85, "summary": "test"}'
])
bad_chain = structured_prompt | bad_model | parser

try:
    bad_chain.invoke({"ticket_id": "T003", "ticket_text": "anything"})
except Exception as e:
    print(f"{type(e).__name__}: {e}")


## 2.4 `with_structured_output` — the approach `chains.py` actually uses in live mode

This is cleaner when your model supports tool calling (Anthropic and OpenAI
chat models do): no manual format instructions, no manual JSON parsing — the
model returns a validated object directly.

```python
from langchain_anthropic import ChatAnthropic
from models import TicketClassification

llm = ChatAnthropic(model="claude-sonnet-5", max_tokens=1000)
structured_llm = llm.with_structured_output(TicketClassification)

result = structured_llm.invoke("ticket_id: T004\nticket_text: My package arrived damaged.")
print(result)   # a TicketClassification instance, no parser needed
```

`FakeListChatModel` doesn't implement tool calling, so this specific method
can't be demonstrated offline — that's exactly why `chains.py` branches on
`MODE` between a `RunnableLambda`-wrapped mock function and this real
structured-output path. You've now seen both halves of that branch.

## Exercise

1. Build a `PydanticOutputParser` for the project's `ValidationResult` model
   (from `models.py`).
2. Write a fake JSON response that matches its schema and run it through a
   `prompt | fake_model | parser` chain like above.
3. Now deliberately break the JSON (wrong type for `is_accurate`, e.g. the
   string `"yes"` instead of `true`) and observe the validation error —
   this is the same failure mode `pipeline.py` is guarding against when it
   validates the classification chain's output before letting anything
   downstream see it.